In [2]:
import time
from pyspark.sql import SparkSession 
from pyspark.sql import functions as F 
from pyspark.sql import types as T
from kafka import KafkaProducer

spark = SparkSession \
    .builder \
    .master("local") \
    .appName("ex6_Real-time_Review") \
    .config('spark.jars.packages', 'org.apache.spark:spark-sql-kafka-0-10_2.12:3.4.0') \
    .getOrCreate()

:: loading settings :: url = jar:file:/usr/local/lib/python3.10/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/developer/.ivy2/cache
The jars for the packages stored in: /home/developer/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-739e791f-2f0e-4807-be0b-113544b37d54;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.4.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.4.0 in central
	found org.apache.kafka#kafka-clients;3.3.2 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.9.1 in central
	found org.slf4j#slf4j-api;2.0.6 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 441ms :: artifacts dl 13m

In [3]:
stream_df = spark \
.readStream \
.format("kafka") \
.option("kafka.bootstrap.servers", "course-kafka:9092") \
.option("subscribe", "gps-with-reviews") \
.option("startingOffsets", "earliest") \
.load() \
.select(F.col("value").cast("string"))

In [4]:
#DataFrame to Console
query = stream_df \
    .writeStream \
    .format("console") \
    .option("truncate", "false") \
    .start()

26/08/14 00:54:11 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-0827e12a-c068-4882-86df-9841bac48f2b. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/08/14 00:54:11 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/08/14 00:54:12 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


-------------------------------------------
Batch: 0
-------------------------------------------
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|value                                                                                                                                                                                                                                                                                                                                                                                                                        |
+----------------------------------------------------------------------

In [5]:
json_schema = T.StructType([
    T.StructField("application_name", T.StringType(), True),
    T.StructField("num_of_positive_sentiments", T.IntegerType()),
    T.StructField("num_of_negative_sentiments", T.IntegerType()),
    T.StructField("num_of_neutral_sentiments", T.IntegerType()),
    T.StructField("avg_sentiment_polarity", T.DoubleType()),
    T.StructField("avg_sentiment_subjectivity", T.DoubleType()),
    T.StructField("category", T.StringType()),
    T.StructField("rating", T.DoubleType()),
    T.StructField("reviews", T.StringType()),
    T.StructField("size", T.IntegerType()),
    T.StructField("num_of_installs", T.DoubleType()),
    T.StructField('price', T.DoubleType()),
    T.StructField('age_limit', T.LongType()),
    T.StructField('genres', T.StringType()), 
    T.StructField('version', T.StringType())
])


In [6]:
#parse data
parsed_df = stream_df \
.withColumn('parsed_json', F.from_json(F.col("value"), json_schema)) \
.select(F.col('parsed_json.*'))

In [11]:
query = parsed_df \
    .writeStream \
    .trigger(processingTime='1 minute') \
    .format('parquet') \
    .outputMode('append') \
    .option("path", "s3a://pyspark/data/target/google_reviews_calc") \
    .option('checkpointLocation', 's3a://pyspark/checkpoints/ex6/store_result') \
    .start()


26/08/14 00:56:37 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


Py4JJavaError: An error occurred while calling o77.start.
: java.lang.IllegalStateException: Cannot call methods on a stopped SparkContext.
This stopped SparkContext was created at:

org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:58)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance0(Native Method)
java.base/jdk.internal.reflect.NativeConstructorAccessorImpl.newInstance(NativeConstructorAccessorImpl.java:62)
java.base/jdk.internal.reflect.DelegatingConstructorAccessorImpl.newInstance(DelegatingConstructorAccessorImpl.java:45)
java.base/java.lang.reflect.Constructor.newInstance(Constructor.java:490)
py4j.reflection.MethodInvoker.invoke(MethodInvoker.java:247)
py4j.reflection.ReflectionEngine.invoke(ReflectionEngine.java:374)
py4j.Gateway.invoke(Gateway.java:238)
py4j.commands.ConstructorCommand.invokeConstructor(ConstructorCommand.java:80)
py4j.commands.ConstructorCommand.execute(ConstructorCommand.java:69)
py4j.ClientServerConnection.wai

In [ ]:
query.awaitTermination()

In [9]:
spark.stop()